# ⚖️ AI Ethics & Governance — Demo
Case study: Loan Approval Model
1. Generate biased data → Train model
2. Bias audit (8 metrics) → Remediation
3. Explainability (SHAP, counterfactuals)
4. EU AI Act classification
5. Model card + Audit trail

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np, pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

## 1. Data & Model

In [ ]:
from data.generate_loan_data import generate_loan_dataset
df = generate_loan_dataset(n_samples=5000)
print(f'Approval by race:\n{df.groupby("race")["approved"].mean()}')
print(f'\nApproval by gender:\n{df.groupby("gender")["approved"].mean()}')

In [ ]:
feat_cols = ['income','credit_score','debt_to_income','employment_years','loan_amount']
X, y = df[feat_cols].values, df['approved'].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
model = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
model.fit(X_tr, y_tr)
y_pred = model.predict(X_te)
y_prob = model.predict_proba(X_te)[:,1]
print(f'Accuracy: {accuracy_score(y_te,y_pred):.3f}, AUC: {roc_auc_score(y_te,y_prob):.3f}')

## 2. Bias Audit

In [ ]:
from app.bias.detector import run_bias_audit
sensitive = (df.iloc[-len(y_te):]['race']=='white').values.astype(int)
report = run_bias_audit(y_te, y_pred, sensitive, 1, 'race', y_prob)
print('\nMetrics:')
for k,v in report.metrics.items(): print(f'  {k}: {v:.4f}')
print(f'\nFlags ({len(report.flags)}):')
for f in report.flags: print(f'  ⚠️ {f}')

In [ ]:
from app.viz.charts import bias_radar_chart, bias_comparison_bar
fig = bias_radar_chart(report.metrics, 'Fairness — Race')
fig.show()

## 3. Remediation

In [ ]:
from app.bias.remediation import compute_reweighing_weights
w = compute_reweighing_weights(y_tr, (df.iloc[:len(y_tr)]['race']=='white').values.astype(int), 1)
model2 = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
model2.fit(X_tr, y_tr, sample_weight=w)
y_pred2 = model2.predict(X_te)
report2 = run_bias_audit(y_te, y_pred2, sensitive, 1, 'race')
print('After remediation:')
for k,v in report2.metrics.items(): print(f'  {k}: {v:.4f}')

In [ ]:
fig = bias_comparison_bar(report.metrics, report2.metrics)
fig.show()

## 4. Explainability

In [ ]:
from app.explainability.counterfactuals import find_counterfactuals
denied = np.where(y_pred==0)[0]
if len(denied)>0:
    cf = find_counterfactuals(model, X_te[denied[0]], feat_cols, target_class=1)
    print(f'Original: DENIED. Found {cf["n_found"]} counterfactuals:')
    for i,c in enumerate(cf['counterfactuals']):
        print(f'  CF{i+1}: change {c["n_features_changed"]} features')
        for ch in c['changes']: print(f'    {ch["feature"]}: {ch["original"]} → {ch["counterfactual"]}')

## 5. EU AI Act

In [ ]:
from app.compliance.eu_ai_act_classifier import classify_risk
result = classify_risk('AI system for credit scoring and loan approval decisions')
print(f'Risk: {result.risk_level.upper()}')
print(f'Domain: {result.matched_domain}')
print(f'Obligations ({len(result.obligations)}):')
for o in result.obligations[:5]: print(f'  • {o}')

## 6. Model Card

In [ ]:
from app.compliance.model_card_generator import generate_model_card
card = generate_model_card('LoanApproval_v1', model_type='GBM',
    description='Loan approval binary classifier',
    metrics={'accuracy': float(accuracy_score(y_te,y_pred)), 'auc_roc': float(roc_auc_score(y_te,y_prob))},
    bias_report=report.to_dict(), compliance_result=result.to_dict(),
    framework='scikit-learn', developers=['Andrea Di Palma'])
print(card.to_markdown()[:800])

## 7. Audit Trail

In [ ]:
from app.audit.trail import AuditTrail
trail = AuditTrail('/tmp/demo_audit.db')
trail.log_event('bias_audit', report.to_dict(), 'LoanApproval_v1', 'Andrea')
trail.log_event('compliance_check', result.to_dict(), 'LoanApproval_v1', 'Andrea')
trail.log_event('remediation', {'method': 'reweighing', 'metrics_after': report2.to_dict()}, 'LoanApproval_v1')
print(f'Events: {trail.count_events()}')
print(f'Integrity: {trail.verify_integrity()}')
print('=== DONE ===')